In [160]:
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.keys import Keys
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.chrome.options import Options
# import re
# import time
# import pandas as pd
# from datetime import datetime

In [161]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import re
import time
import pandas as pd
from datetime import datetime

In [162]:
username = "elonmusk"  # account om te scrapen
end_date = datetime.today().strftime("%Y-%m-%d")
scroll_pause = 2  # seconden tussen scrolls


In [163]:
my_twitter_email = "Luiz.verheyen@icloud.com"
my_twitter_username = "verheyen_l89639"
my_twitter_password = "Citroen3012"

In [164]:
options = uc.ChromeOptions()
options.add_argument("--start-maximized")
# options.add_argument("--disable-notifications")
# opties zoals headless kan ook
# options.add_argument("--headless")

In [165]:
driver = uc.Chrome(version_main=143, options=options)
time.sleep(5)

In [166]:
# Ga naar loginpagina
driver.get("https://twitter.com/login")
time.sleep(5)  # wacht tot de pagina volledig geladen is

In [167]:
username_input = driver.find_element(By.NAME, "text")
username_input.send_keys(my_twitter_username)
time.sleep(1)
username_input.send_keys(Keys.ENTER)
time.sleep(3)

In [168]:
try:
    username_input = driver.find_element(By.NAME, "text")
    username_input.send_keys(my_twitter_email)
    time.sleep(1)
    username_input.send_keys(Keys.ENTER)
    time.sleep(3)
except:
    print("Geen username nodig om in te loggen")

Geen username nodig om in te loggen


In [169]:
try:
    password_input = driver.find_element(By.NAME, "password")
    password_input.send_keys(my_twitter_password)
    time.sleep(5)
    password_input.send_keys(Keys.ENTER)
    time.sleep(5)
except:
    print("Geen wachtwoord invoer vereist of al ingelogd.")

In [170]:
driver.get(f"https://twitter.com/{username}")
time.sleep(7)  # wachten tot pagina laadt

In [171]:
try:
    cookie_button = driver.find_element(By.XPATH, '//button[contains(., "Accept all cookies")]')
    cookie_button.click()
    print("Cookies geaccepteerd.")
    time.sleep(2)
except:
    print("Geen cookie-wall gevonden of al geaccepteerd.")

Cookies geaccepteerd.


In [172]:
tweets_data = []

In [173]:
def human_scroll(driver, total_scroll=3000, step=300, pause=0.3):
    scrolled = 0
    while scrolled < total_scroll:
        driver.execute_script(f"window.scrollBy(0, {step});")
        scrolled += step
        time.sleep(pause)

In [174]:
# Instellingen
target_date = end_date  # Stop met scrapen bij tweets ouder dan 1 jan 2026
tweets_ids = set() 
stop_scraping = False

print(f"Start met scrapen tot aan {target_date}...")

while stop_scraping == False:
    if stop_scraping:
        break
        
    # Zoek alle zichtbare tweets
    articles = driver.find_elements(By.XPATH, '//article[@role="article" and @data-testid="tweet"]')
    
    for article in articles:
        try:
            # 1. Datum ophalen (Cruciaal voor de stop-check)
            time_elem = article.find_element(By.XPATH, './/time')
            date_str = time_elem.get_attribute("datetime")
            # Converteer naar offset-naive datetime voor vergelijking
            tweet_date = datetime.fromisoformat(date_str.replace("Z", "+00:00")).replace(tzinfo=None)
            

            # 2. Unieke ID check (voorkom dubbele regels in CSV)
            tweet_id = article.find_element(By.XPATH, './/time/..').get_attribute('href')
            tweet_username = tweet_id.split("/")[3]
            
            
            if tweet_username == username:
                
                if tweet_date.strftime("%Y-%m-%d") != target_date:
                    print(f"tweet with date {tweet_date} found, exiting")
                    stop_scraping = True
                    break
            
                if tweet_id in tweets_ids:
                    continue

                # 3. Tekst en Statistieken ophalen
                text = article.find_element(By.XPATH, './/div[@data-testid="tweetText"]').text
                
                # Statistieken uit de aria-label
                stats_group = article.find_element(By.XPATH, './/div[@role="group"]')
                label = stats_group.get_attribute("aria-label")
                
                def parse_stat(pattern, text):
                    match = re.search(pattern, text.lower())
                    if match:
                        return int(re.sub(r'[^\d]', '', match.group(1)))
                    return 0

                replies = parse_stat(r'(\d[\d\.,]*)\s+replies', label)
                reposts = parse_stat(r'(\d[\d\.,]*)\s+reposts', label)
                likes = parse_stat(r'(\d[\d\.,]*)\s+likes', label)
                bookmarks = parse_stat(r'(\d[\d\.,]*)\s+bookmarks', label)
                views = parse_stat(r'(\d[\d\.,]*)\s+views', label)

                tweets_data.append([tweet_date, tweet_username, text, replies, reposts, likes, bookmarks, views])
                tweets_ids.add(tweet_id)
                print(f"Opgeslagen: {tweet_date.strftime('%Y-%m-%d %H:%M')} | @{tweet_username} | Likes: {likes}")

        except Exception:
            continue
    
    # Scroll naar beneden om nieuwe tweets te laden
    human_scroll(driver, total_scroll=2500, step=250, pause=0.4)
    time.sleep(scroll_pause)

print(f"Klaar! {len(tweets_data)} tweets verzameld.")

Start met scrapen tot aan 2026-01-23...
Opgeslagen: 2026-01-23 07:18 | @elonmusk | Likes: 19690
Opgeslagen: 2026-01-23 07:37 | @elonmusk | Likes: 7558
Opgeslagen: 2026-01-23 07:25 | @elonmusk | Likes: 107167
Opgeslagen: 2026-01-23 07:22 | @elonmusk | Likes: 33530
Opgeslagen: 2026-01-23 07:17 | @elonmusk | Likes: 52034
Opgeslagen: 2026-01-23 07:14 | @elonmusk | Likes: 32047
Opgeslagen: 2026-01-23 03:56 | @elonmusk | Likes: 145623
Opgeslagen: 2026-01-23 03:35 | @elonmusk | Likes: 34199
Opgeslagen: 2026-01-23 03:33 | @elonmusk | Likes: 49690
Opgeslagen: 2026-01-23 03:28 | @elonmusk | Likes: 8427
Opgeslagen: 2026-01-23 03:25 | @elonmusk | Likes: 26974
Opgeslagen: 2026-01-23 03:19 | @elonmusk | Likes: 46757
Opgeslagen: 2026-01-23 03:17 | @elonmusk | Likes: 25940
Opgeslagen: 2026-01-23 03:16 | @elonmusk | Likes: 7447
Opgeslagen: 2026-01-23 03:12 | @elonmusk | Likes: 16443
Opgeslagen: 2026-01-23 03:02 | @elonmusk | Likes: 124754
Opgeslagen: 2026-01-23 02:44 | @elonmusk | Likes: 31830
Opgeslag

In [175]:
df = pd.DataFrame(tweets_data, columns=["Date", "Username", "Content", "Replies", "Reposts", "Likes", "Bookmarks", "Views"])
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d %H:%M')
df['Time'] = pd.to_datetime(df['Date']).dt.time
df['Date'] = pd.to_datetime(df['Date']).dt.date
df.to_csv(f"{username}_tweets.csv", index=False)
print(f"{len(df)} tweets opgeslagen!")

33 tweets opgeslagen!


In [176]:
df

,Date,Username,Content,Replies,Reposts,Likes,Bookmarks,Views,Time
0,2026-01-23,elonmusk,Davos discussion,3166,4264,19690,5542,9427205,07:18:00
1,2026-01-23,elonmusk,Join x/xAI!,1634,1628,7558,554,1470668,07:37:00
2,2026-01-23,elonmusk,Yes,3147,18086,107167,3275,4299584,07:25:00
3,2026-01-23,elonmusk,"Best rocket engine ever:\n\nFull-flow, staged ...",1973,4023,33530,1267,2652431,07:22:00
4,2026-01-23,elonmusk,Been going on a long time,1495,11994,52034,2467,3013050,07:17:00
5,2026-01-23,elonmusk,What is this blue slip bs?,2552,9341,32047,1356,3642902,07:14:00
6,2026-01-23,elonmusk,Our judicial system is broken,5085,20893,145623,1540,13512305,03:56:00
7,2026-01-23,elonmusk,,758,2626,34199,410,23486373,03:35:00
8,2026-01-23,elonmusk,Good question,1248,8095,49690,422,10221682,03:33:00
9,2026-01-23,elonmusk,Grok can understand video,1100,1528,8427,403,1363397,03:28:00
